# 4 · AI Workflows & System Design for Finance
**You leave with:** a Company Screening Engine — a program that pulls fundamentals for 16 companies live from the U.S. Securities and Exchange Commission (SEC), keeps the ones that pass two tests you control (revenue growth of at least 8%, net margin of at least 10%), asks a large language model (LLM) to write a short case for each shortlisted company, and machine-checks every number in those sentences — finished as a LangGraph workflow, the form professional teams use for pipelines like this.

**The storyline so far.** In Session 2 you valued Apple against seven peers; the data arrived live from SEC filings through a loader the course had built for you, for that one fixed peer list. In Session 3 you learned to prove such a number right. Today you assemble the machinery yourself: the fetch, the filter, the audit — pointed at 16 companies now, and at any companies you choose tomorrow. This is the session where your course work becomes a **tool** — the one you will publish on GitHub in Session 5.

**In this notebook you will:**

- Read the raw filing record EDGAR returns for a ticker, and turn it into one clean row of metrics
- Fetch 16 companies and screen them: revenue growth of at least 8% AND net margin of at least 10% — then ask whether Apple passes its own screen
- Have the model write a short rationale for each shortlisted company, and trace every number in it back to your table
- Train a simple model that forecasts next year's revenue, and measure its error honestly
- Rebuild the whole pipeline as a LangGraph workflow graph


## The pattern of the day

```
INPUT → RETRIEVE → STRUCTURE → REASON → VALIDATE → HUMAN
         (code)     (code)     (model)    (code)    (you)
```

A **workflow** is a fixed plan written by you; the model fills designated steps. Three design rules carry everything:

1. **Code does math; the model does judgment.** Growth rates and filters live in pandas; the model writes grounded prose *about* them.
2. **Validate at the boundary.** Schema-forced output, plus a numeric audit: any figure in the prose that doesn't trace to your inputs gets flagged.
3. **The human gate is the exit.** Nothing is saved or sent without approval.

## SEC EDGAR in brief

EDGAR (Electronic Data Gathering, Analysis and Retrieval) is the U.S. Securities and Exchange Commission's public filing database.

Every US-listed company's filings, free, no key: just identify yourself (`SEC_EDGAR_USER_AGENT` in `.env`). **There:** 10-K/10-Q/8-K/20-F documents + XBRL fundamentals. **Not there:** prices, estimates. Complications we met while building this course (details: `session-04-workflows/edgar-cheatsheet.md`): companies drift between XBRL tags across years; foreign filers report IFRS in local currency; most pharma tags **no operating income** at all.

In [ ]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# If you pulled course updates while this kernel was running, pick them up here
# (a no-op on a fresh kernel; saves a restart otherwise):
import importlib
for _n in [n for n in list(sys.modules) if n.startswith("toolkit")]:
    importlib.reload(sys.modules[_n])
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"repo root: {ROOT}")
print(f"API key:   {'configured' if HAS_KEY else 'NOT SET - cells that call Claude will be skipped'}")

## Part A: read the raw filing record

EDGAR (Electronic Data Gathering, Analysis and Retrieval) is the SEC's public filing database: every listed company's official reports, free and machine-readable.

You have already used it: Session 2's loader fetched Apple and its peers from EDGAR for you. What that loader was reading is the record below — run the cell and study it, because everything in this session is built from records like this one.

In [ ]:
from toolkit import edgar

print("Loading data live from SEC EDGAR (data.sec.gov) ...")
fin = edgar.annual_financials("AAPL", n=3)
print(json.dumps(fin, indent=1))   # the whole record: this is your raw material

In [ ]:
# That's a real 10-K speaking. Let's compute metrics IN CODE (rule 1):
def metrics_row(ticker: str) -> dict | None:
    """3-year fundamentals -> one screening row. Returns None if unusable."""
    try:
        fin = edgar.annual_financials(ticker, n=3)
    except edgar.EdgarError as e:
        print(f"  {ticker}: skipped ({e})")
        return None
    rev = fin["revenue"]
    ni = {v["fy_end"]: v["val"] for v in fin["net_income"]}
    if len(rev) < 3 or fin["unit"] != "USD":
        return None
    r0, r1, r2 = (rev[i]["val"] for i in range(3))
    n_now, n_prev = ni.get(rev[2]["fy_end"]), ni.get(rev[1]["fy_end"])
    return {"ticker": fin["ticker"], "company": fin["company"], "fy_end": rev[2]["fy_end"],
            "revenue_bn": round(r2 / 1e9, 2),
            "growth_1y": round(r2 / r1 - 1, 4),
            "cagr_2y": round((r2 / r0) ** 0.5 - 1, 4),
            "net_margin": round(n_now / r2, 4) if n_now is not None else None,
            "net_margin_prior": round(n_prev / r1, 4) if n_prev is not None else None}

metrics_row("AAPL")

## Part B: LAB: the Company Screening Engine

The engine answers one question: **of these 16 companies, which grew revenue by at least 8% last year while keeping at least 10 cents of every revenue dollar as profit?** Both thresholds are parameters — you will change them and watch the shortlist move.

Universe: 16 tickers, listed in `session-04-workflows/data/universe.csv` — Apple and its seven Session 2 peers, plus four pharmaceutical and four industrial companies, so the screen faces sectors with different economics. **The CSV supplies only the ticker list; every metric is fetched live from EDGAR.** First run fetches ~16 filings (a minute or two); everything caches.

One question to carry through the lab: **does Apple pass its own screen?**

### Exercise 1: fetch the universe

In [ ]:
import pandas as pd
universe = pd.read_csv(ROOT / "session-04-workflows" / "data" / "universe.csv")   # tickers only - every metric comes live from EDGAR

### START CODE HERE ###
rows = [m for t in universe[None]                     # which column holds the tickers?
        if (m := metrics_row(None)) is not None]      # call the fetcher with what?
df = pd.DataFrame(None)                               # build the table from what?
### END CODE HERE ###

print(f"{len(df)} of {len(universe)} companies fetched")
df.round(3)

In [ ]:
# ✅ self-check: run me
assert len(df) >= 14, "expected at least 14 of 16 to fetch - is your loop skipping None rows?"
assert {"growth_1y", "net_margin"} <= set(df.columns)
assert df["net_margin"].abs().max() < 1, "margins should be decimals, not percents"
print("All checks passed ✅")

### Exercise 2: the deterministic screen
Filter: `growth_1y >= min_growth` AND `net_margin >= min_margin`; if `require_improving`, also `net_margin > net_margin_prior`. Sort by growth, descending.

In [ ]:
def apply_screen(df, min_growth=0.08, min_margin=0.10, require_improving=False):
### START CODE HERE ###
    mask = (df[None] >= min_growth) & (df[None] >= min_margin)
    if require_improving:
        mask &= df["net_margin"] > df[None]               # improving vs which column?
    return df[mask].sort_values(None, ascending=False)    # rank the shortlist by what?
### END CODE HERE ###

shortlist = apply_screen(df)
print("Shortlist:", ", ".join(shortlist["ticker"]))
shortlist.round(3)

In [ ]:
# ✅ self-check: run me
assert len(shortlist) >= 1, "empty shortlist with default criteria - check your mask logic"
assert (shortlist["growth_1y"] >= 0.08).all() and (shortlist["net_margin"] >= 0.10).all()
tight = apply_screen(df, min_growth=0.10, require_improving=True)
assert len(tight) <= len(shortlist), "tighter criteria cannot grow the shortlist"
print("All checks passed ✅  Now play: change the criteria and watch the shortlist move.")

**The screen's first finding.** Look for Apple in the shortlist: it is not there. With the default criteria, Apple **fails its own peer screen** — its latest-year revenue growth is about 6%, below the 8% dial, while ten of its sixteen neighbors pass. Now recall Session 2: the market prices Apple at roughly a 60% premium to what its peers' ratings imply. Hold both facts at once: **the premium is not in the trailing fundamentals your screen reads — it is the market's priced conviction about the future.** A screen tells you what the filings say; it does not tell you what the market believes. Rerun `apply_screen(df, min_growth=0.05)` and watch Apple enter: criteria are dials, and knowing what each dial excludes is the analyst's job.

### Exercise 3: the written case, grounded and audited

**The goal.** A shortlist is numbers; an investment committee reads prose. This exercise turns each shortlisted company's metrics into a short written case — and makes that prose as accountable as the numbers behind it.

Two mechanisms carry the trust:

1. **Grounding.** The model receives *only* the metrics table you send between `<metrics>` tags, with instructions to use nothing else. Every claim in the case must therefore come from your own table — there is nothing else in the prompt to draw on.
2. **A validated contract.** The output shape is declared with **Pydantic**: a class per object, a typed field per key. `llm.ask_pydantic` forces the reply through that schema — it either parses into typed `Rationale` objects or is rejected at the boundary, never passed along malformed. The cell's last line states the validation explicitly: `assert isinstance(result, RationaleSet)`.

The **numeric audit** then closes the loop (`verify.novel_numbers`): every figure in the prose is traced back to your table, and anything untraceable is flagged ⚠️.

**Read the flags as a shortlist, not an accusation.** A clean run typically flags a few figures anyway, and they are usually the model's own derived ratios: "roughly 3.5 times", "a gap of 6 percentage points". The checker compares against the numbers you supplied; it cannot perform derivations, so it cannot distinguish a legitimate calculation from an invention. It hands both to you. That is the honest boundary of this kind of automation, and it is still valuable: it reduces what you must verify from an entire page of prose to three or four numbers.

In [ ]:
from pydantic import BaseModel
from toolkit import llm, verify

class Rationale(BaseModel):
    ticker: str
    observation: str

class RationaleSet(BaseModel):
    rationales: list[Rationale]

if HAS_KEY and len(shortlist):
### START CODE HERE ###
    result = llm.ask_pydantic(
        "Write a 2-3 sentence investment observation per company from these screening "
        "metrics (growth/margins as decimals). Use ONLY these metrics - no outside "
        f"knowledge, no new numbers:\n<metrics>\n{shortlist.to_json(orient='records')}\n</metrics>",
        None,                                             # force which output model?
        system="You are screening companies for an investment committee. Grounded, direct, no hype.")
    source_vals = [v for r in shortlist.to_dict("records") for v in r.values()
                   if isinstance(v, (int, float))]
    for r in result.rationales:
        flags = verify.novel_numbers(None, None)          # audit WHICH text against WHICH numbers?
        mark = f"  ⚠️ untraceable: {flags}" if flags else "  ✅ grounded"
        llm.show(f"{r.observation}{mark}", title=r.ticker)
### END CODE HERE ###
    assert isinstance(result, RationaleSet), "typed at the boundary: this is what Pydantic buys you"
    print(f"\nEvery rationale arrived as a typed Rationale object. Token usage: {llm.usage_summary()}")
else:
    print("No API key (or empty shortlist) - the deterministic screen above is still the deliverable core.")

### Exercise 4: train a model to forecast revenue

So far the AI in this course has been **generative**: a language model producing text. The other half of AI is **predictive**: a model whose parameters are *trained* on observed data, and whose quality is *measured* on data it has not seen. This exercise uses the simplest trainable model there is, a least-squares trend line, on six years of real revenue from the filings.

The discipline is the entire lesson, and it transfers unchanged to any model, however sophisticated:

1. **Train** on every year except the last.
2. **Test** on the held-out last year: the error against a value the model never saw is your honest measure of quality.
3. Only then **retrain on everything and forecast** the next year.

This discipline is also the difference from asking the language model to forecast: an LLM's guess cannot be backtested per run; a trained model can.

In [ ]:
import numpy as np

def forecast_revenue(revenues):
    """Least-squares trend: train without the last year, measure on it, then forecast.

    Returns (holdout_error, next_year_forecast)."""
    years = np.arange(len(revenues), dtype=float)
    rev = np.asarray(revenues, dtype=float)
### START CODE HERE ###
    slope, intercept = np.polyfit(years[None], rev[None], 1)   # train WITHOUT the last year
    predicted_last = slope * years[-1] + intercept              # predict the held-out year
    holdout_error = abs(None - rev[-1]) / rev[-1]               # |prediction - truth| / truth
    slope, intercept = np.polyfit(years, rev, 1)                # now retrain on ALL years
    next_forecast = slope * (None + 1) + intercept              # one year beyond the last
### END CODE HERE ###
    return holdout_error, next_forecast

In [ ]:
# ✅ self-check: run me (offline). On perfectly linear data the model must be near-perfect.
err, fc = forecast_revenue([100, 110, 120, 130, 140, 150])
assert err < 0.01, "on linear data the holdout error must be ~0 - did you train WITHOUT the last year?"
assert abs(fc - 160) < 1, "the next value on a +10-per-year trend is 160"
print("All checks passed ✅  A trained, tested, honest forecaster. Now real companies:")

In [ ]:
for t in ["AAPL", "ETN", "NVDA", "GE"]:
    revs = [v["val"] for v in edgar.annual_financials(t, n=6)["revenue"]]
    err, fc = forecast_revenue(revs)
    print(f"{t:5} last actual {revs[-1]/1e9:7,.1f}bn | holdout error {err:5.0%} | "
          f"FY+1 forecast {fc/1e9:7,.1f}bn {'<- error too large to trust this forecast' if err > 0.15 else ''}")

In [ ]:
# The same four companies, drawn: six actual fiscal years and the FY+1 forecast.
import plotly.graph_objects as go
from plotly.subplots import make_subplots

series = {}
for t in ["AAPL", "ETN", "NVDA", "GE"]:
    recs = edgar.annual_financials(t, n=6)["revenue"]
    revs = [v["val"] for v in recs]
    err, fc = forecast_revenue(revs)
    series[t] = ([int(v["fy_end"][:4]) for v in recs], revs, err, fc)

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f"{t} — holdout error {v[2]:.0%}" for t, v in series.items()])
for i, (t, (years, revs, err, fc)) in enumerate(series.items()):
    r, c = divmod(i, 2)
    fig.add_trace(go.Scatter(x=years, y=[x / 1e9 for x in revs], mode="lines+markers",
                             showlegend=False), row=r + 1, col=c + 1)
    fig.add_trace(go.Scatter(x=[years[-1], years[-1] + 1], y=[revs[-1] / 1e9, fc / 1e9],
                             mode="lines+markers", line=dict(dash="dot"),
                             marker=dict(symbol="diamond"), showlegend=False), row=r + 1, col=c + 1)
fig.update_layout(height=560, title_text="Revenue in $bn: actual (solid) and the FY+1 forecast (dotted)")
fig.show()

**Observation.** The error column carries more information than the forecast column, and each row teaches a different lesson (your exact numbers will vary as new filings arrive):

- **Apple and Eaton** come out with **single-digit errors**: large, maturing businesses in a roughly linear regime, where this model class applies and the forecast is usable. Note what the forecast says about Apple: continuation, not acceleration — consistent with what your screen found above.
- **NVIDIA** comes out around **40%**: a straight line cannot describe exponential growth. The model is not broken; it is honestly reporting that it is the wrong model class for this company.
- **General Electric** comes out worst of all, and no model fixes it: GE spun off its healthcare and energy businesses, so the six revenues are not one company's history. **The premise failed before the model ran** — exactly the kind of fact a fitted line cannot know and an analyst must.

Three professional conclusions carry beyond this exercise:

- **Never quote a forecast without its measured error**; report the two as a pair.
- **A large error is information**: it says the growth regime is nonlinear, or the data is not what you assumed. Model selection starts from measured failure.
- **Prediction and generation divide the labor.** The trained model produces the number and its error; the language model, in the exercise above, explains screened companies in grounded prose. Neither should do the other's job.

## Part C: the same pipeline, in the industry's language

What you built in Part B — fetch, screen, forecast, reason, audit — is a pipeline of plain Python functions, held together by the order of your notebook cells. Professional teams write the same thing as an explicit **graph**: each step is a node, each arrow is an edge, and a library runs the graph, records every state transition, and can pause, resume or retry it.

The most common choice is **LangGraph**, part of the LangChain ecosystem — one of the most widely used families of tools for building applications around language models. Nothing about your logic changes: LangGraph does not care how a node does its work, so every node below reuses a function you already wrote and the model call goes through the same official SDK you have used all course.

**Why LangGraph.** For ten lines of linear Python it is not worth it — plain functions in order are simpler, and knowing that is part of the lesson. The graph earns its place when a workflow has any of these four things:

1. **An unreliable step in the loop.** The rationale node is a model call: it can time out, fail, or return output the schema rejects. A graph can retry and resume *at the failed node* instead of rerunning everything before it. The intuition that graphs are about uncertainty is right — and the model node is where the uncertainty lives.
2. **Paths that depend on the data.** Conditional edges route a run: an empty shortlist can end it early; audit flags can send the prose back for one more attempt. A fixed script cannot change route mid-run; a graph can — without handing the plan to the model.
3. **State you must be able to show.** Every transition is recorded as a checkpoint, so "why did it produce this?" has an answer — and a human gate (LangGraph calls it an *interrupt*) can pause the run before anything is saved or sent.
4. **The road to agents.** Session 5's agent is this same graph with one edge that loops back, so the model can decide to act again. Learn the abstraction on a deterministic workflow today and tomorrow's agent holds no mystery.

**In our case the graph pays twice.** First, this pipeline's one unreliable step — the live model call in `rationale` — now sits inside a structure that can retry or resume exactly that node when the engine runs on a schedule, instead of repeating the sixteen fetches before it. Second, this graph is the precise object Session 5 turns into an agent: one loop-back edge is the entire difference. The printed diagram is the visible bonus — the plan, explicit and unchangeable by the model.

### Exercise 5: wire the graph

The five nodes are written for you — each takes the current state and returns the piece it adds. Your job is the part that makes it a workflow: the **edges**, the fixed plan the model cannot change.

In [ ]:
try:
    from langgraph.graph import StateGraph, START, END
except ModuleNotFoundError:          # env set up before LangGraph joined the course
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langgraph", "grandalf"], check=True)
    from langgraph.graph import StateGraph, START, END

from typing import TypedDict

class ScreenState(TypedDict, total=False):
    tickers: list        # input
    table: pd.DataFrame  # after fetch
    shortlist: pd.DataFrame
    forecasts: dict      # ticker -> holdout error and FY+1 forecast
    rationales: list     # typed Rationale objects
    flags: dict          # ticker -> untraceable numbers

def fetch_node(state: ScreenState) -> dict:
    rows = [m for t in state["tickers"] if (m := metrics_row(t)) is not None]
    return {"table": pd.DataFrame(rows)}

def screen_node(state: ScreenState) -> dict:
    return {"shortlist": apply_screen(state["table"], min_growth=0.10, min_margin=0.20)}

def forecast_node(state: ScreenState) -> dict:
    out = {}
    for t in state["shortlist"]["ticker"]:
        revs = [v["val"] for v in edgar.annual_financials(t, n=6)["revenue"]]
        err, fc = forecast_revenue(revs)
        out[t] = {"holdout_error": round(err, 3), "fy1_forecast_bn": round(fc / 1e9, 1)}
    return {"forecasts": out}

def rationale_node(state: ScreenState) -> dict:
    result = llm.ask_pydantic(
        "Write a 2-3 sentence investment observation per company from these screening "
        "metrics (growth/margins as decimals). Use ONLY these metrics - no outside "
        f"knowledge, no new numbers:\n<metrics>\n{state['shortlist'].to_json(orient='records')}\n</metrics>",
        RationaleSet,
        system="You are screening companies for an investment committee. Grounded, direct, no hype.")
    return {"rationales": result.rationales}

def audit_node(state: ScreenState) -> dict:
    vals = [v for r in state["shortlist"].to_dict("records") for v in r.values()
            if isinstance(v, (int, float))]
    return {"flags": {r.ticker: verify.novel_numbers(r.observation, vals)
                      for r in state["rationales"]}}

g = StateGraph(ScreenState)
for name, fn in [("fetch", fetch_node), ("screen", screen_node), ("forecast", forecast_node),
                 ("rationale", rationale_node), ("audit", audit_node)]:
    g.add_node(name, fn)

g.add_edge(START, "fetch")
### START CODE HERE ###
g.add_edge("fetch", None)         # after fetching, which step runs?
g.add_edge("screen", None)        # the deterministic forecaster runs on the shortlist
g.add_edge(None, "rationale")     # the model reasons once all the numbers are in
g.add_edge("rationale", None)     # what must happen to prose before a human sees it?
### END CODE HERE ###
g.add_edge("audit", END)

workflow = g.compile()
print(workflow.get_graph().draw_ascii())

In [ ]:
# ✅ self-check: run me
edges = {(e.source, e.target) for e in workflow.get_graph().edges}
assert ("fetch", "screen") in edges, "the screen must run on fetched data"
assert ("screen", "forecast") in edges, "the forecaster runs on the shortlist, after screening"
assert ("forecast", "rationale") in edges, "the model reasons once all the numbers - forecasts included - are in"
assert ("rationale", "audit") in edges, "prose is audited before any human reads it"
print("All checks passed ✅  The plan is now explicit, drawn, and unchangeable by the model.")

In [ ]:
if HAS_KEY:
    final = workflow.invoke({"tickers": list(universe["ticker"])})
    print("Shortlist:", ", ".join(final["shortlist"]["ticker"]), "\n")
    for t, f in final["forecasts"].items():
        print(f"{t:6} FY+1 forecast {f['fy1_forecast_bn']:8,.1f}bn | holdout error {f['holdout_error']:.0%}")
    print()
    for r in final["rationales"]:
        flags = final["flags"][r.ticker]
        mark = f"  ⚠️ untraceable: {flags}" if flags else "  ✅ grounded"
        llm.show(f"{r.observation}{mark}", title=r.ticker)
    print(f"\nSame engine, same audit - now one `workflow.invoke()` call. Token usage: {llm.usage_summary()}")
else:
    print("No API key - the graph above is still the deliverable: the plan, made explicit.")

**What just happened, mapped.** Every LangGraph term names something you built by hand this session:

| Your Part B | LangGraph |
|---|---|
| the order of your notebook cells | the graph and its edges |
| variables passed between cells | the typed state (`ScreenState`) |
| each function | a node |
| running the cells top to bottom | `workflow.invoke()` |
| the human gate | `interrupt()` before a node (Session 5) |

The full 5-step terminal version of this workflow (with memo rendering and the approval gate) lives in `session-04-workflows/demo/market_intel_workflow.py`; the dry-run requires no key. A worthwhile experiment: open `session-04-workflows/data/example_intel_memo.json`, alter one figure, and rerun with `--dry-run`; the validator flags the altered value. Restore the file afterwards with `git checkout`.

## Push today's work to your repository

The engine belongs in the repository you created in Session 2 (`my-finance-toolkit`). In the **terminal**:

```bash
git add -A
git commit -m "Company screening engine: live EDGAR fetch, screen, forecasts, LangGraph workflow"
git push
```

(Repository not created yet? `gh repo create my-finance-toolkit --private --source . --push`, or github.com → New repo → "push an existing repository". Cheatsheet: `cheatsheets/git-github-for-finance.md`.)

## Deliverable checklist

- [ ] All ✅ checks green; you ran the screen with at least two different criteria sets
- [ ] You can answer: does Apple pass its own screen, and what does that mean? Why did we screen on NET margin?
- [ ] (With key) rationales printed with the numeric-audit marks; anything ⚠️ judged by you
- [ ] The LangGraph version drew its own diagram and reproduced the shortlist with its forecasts
- [ ] Committed and pushed to **your own GitHub repository** — the tool you publish in Session 5 is taking shape

**Next:** `05-agents.ipynb`: the model makes the plan, within controls you define.